# CarVision AI - Khảo sát Dữ liệu Huấn luyện (Tuần 1)
## Phân tích Stanford Cars & Lựa chọn 12 Nhóm Xe Thử nghiệm

Notebook này phục vụ nhóm nghiên cứu CarVision AI:
- **Nguồn dữ liệu:** `tanganke/stanford_cars` (Hugging Face / TFDS cars196).
- **Mục tiêu:** Hiểu cấu trúc 196 nhãn fine-grained, chọn lọc 12 nhóm xe phổ biến và phù hợp nhất với thị trường Việt Nam làm tập PoC (Proof of Concept) cho baseline model.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Thêm root directory vào sys.path để import các module trong src/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.stanford_cars_classes import (
    STANFORD_CARS_CLASSES,
    SELECTED_VN_CLASSES,
    parse_class_name
)

print(f"[✓] Đã nạp thành công {len(STANFORD_CARS_CLASSES)} nhãn xe từ Stanford Cars.")

### 1. Phân tích cấu trúc 196 nhãn trong Stanford Cars
Mỗi nhãn trong Stanford Cars bao gồm: `[Hãng xe] [Dòng xe] [Kiểu dáng] [Năm sản xuất]`.

In [ ]:
parsed_data = [parse_class_name(name) for name in STANFORD_CARS_CLASSES]
df_classes = pd.DataFrame(parsed_data)

print("Top 10 Hãng xe có nhiều dòng nhất:")
display(df_classes["make"].value_counts().head(10))

print("\nPhân bố theo kiểu dáng xe (Body Type):")
display(df_classes["body_type"].value_counts())

### 2. Danh sách 12 nhóm xe chọn lọc cho thị trường Việt Nam (Phase 1 PoC)
Do mục tiêu của CarVision AI hướng tới tra cứu thông số có dẫn chứng, nhóm ưu tiên chọn các dòng xe đã hoặc đang lăn bánh phổ biến tại Việt Nam.

In [ ]:
vn_list = []
for label_id, info in SELECTED_VN_CLASSES.items():
    vn_list.append({
        "Label ID": label_id,
        "Tên nguyên bản (Stanford Cars)": info["raw_name"],
        "Hãng": info["make"],
        "Dòng xe": info["model"],
        "Phân khúc": info["segment"],
        "Các phiên bản/đời xe tại VN": ", ".join(info["vn_generations"]),
        "Mức độ phổ biến": info["vn_popularity"]
    })

df_vn = pd.DataFrame(vn_list)
display(df_vn)

### 3. Stream & Xem ảnh mẫu trực tiếp từ Hugging Face
Không cần tải 16.000+ ảnh về máy làm đầy bộ nhớ, chúng ta dùng cơ chế `streaming=True` của Hugging Face.

In [ ]:
from datasets import load_dataset

print("[+] Đang kết nối dataset qua stream...")
ds_stream = load_dataset("tanganke/stanford_cars", split="train", streaming=True)

# Lấy 1 ảnh mẫu của mỗi nhóm trong danh sách chọn lọc
samples = {}
for item in ds_stream:
    lbl = item["label"]
    if lbl in SELECTED_VN_CLASSES and lbl not in samples:
        samples[lbl] = item["image"]
    if len(samples) == len(SELECTED_VN_CLASSES):
        break

print(f"[✓] Đã lấy thành công {len(samples)} ảnh mẫu cho các lớp chọn lọc.")

In [ ]:
# Hiển thị ảnh mẫu dạng lưới (Grid)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, (lbl, img) in enumerate(samples.items()):
    class_info = SELECTED_VN_CLASSES[lbl]
    axes[i].imshow(img)
    axes[i].set_title(f"[{lbl}] {class_info['model']}\n({class_info['segment']})", fontsize=10)
    axes[i].axis("off")

plt.tight_layout()
plt.show()

### 4. Kết luận & Hướng đi kế tiếp cho Tuần 2
1. **Tập huấn luyện PoC:** Giữ nguyên 12 lớp này để huấn luyện `Custom CNN` (baseline) và `MobileNetV2` (transfer learning).
2. **Thông số kỹ thuật:** Tạo file `app/specs/car_specs.json` lưu thông số chi tiết (động cơ, kích thước, tiêu hao, link dẫn chứng) cho 12 nhóm này.
3. **Mở rộng xe Việt Nam:** Bổ sung tập test riêng cho VinFast VF8 / Toyota Vios ở giai đoạn đánh giá độ thích ứng mô hình.